# 10 · Retarget a real TS onto fresh SMILES

Take a DFT transition state, rebuild it around a different catalyst or substrate written as SMILES, holding
the reacting core where the DFT put it, then search the new periphery.

What goes in:

- the TS `.xyz`, used only as a source of coordinates for the core atoms
- the reacting-core indices, 0-based in xyz order
- the fragment SMILES to assemble

Not a SMILES of the TS itself: partial and hypervalent bonds make its perceived SMILES unreliable. The core
comes from the geometry, the molecule from your fragment SMILES, and SMARTS ties the two together.

Two ways to tie them: a whole-fragment match when the molecule is unchanged, and small anchor SMARTS on the
conserved reacting motif when it is not.


In [ ]:
import tempfile

import numpy as np
import rxembed.pipeline as rx
import xyzrender
from IPython.display import display
from rdkit import Chem, RDLogger
from rdkit.Chem import rdMolAlign
from rdkit.Chem import rdMolTransforms as T

RDLogger.DisableLog("rdApp.*")
rx.set_verbose("INFO")

ORIG, NEW = "#5b8fd6", "#e0736b"  # original TS (blue) vs retargeted molecule (red)


def overlay(ens, ref, ref_xyz, atom_map):
    """Superpose the lowest conformer on the reference by its grafted core, and draw both.

    `atom_map` is [(atom in ens, atom in ref)]. Geometric alignment on the core only, so the periphery is
    free to differ visibly, which is the point of the picture.
    """
    low = ens.lowest(1)
    rdMolAlign.AlignMol(low.mol, ref, prbCid=low.ids[0], refCid=0, atomMap=atom_map)
    return display(
        xyzrender.render(
            xyzrender.load(ref_xyz),
            mol_color=ORIG,
            overlay=low.dump(tempfile.mktemp(suffix=".xyz")),
            overlay_color=NEW,
            auto_align=False,
            no_hy=True,
        )
    )

## 1 · Rebuild the TS from its two fragments

A chiral phosphoric acid TS, 103 atoms: a proton in flight between the phosphate and the substrate carbonyl
with a forming C-C closure. Both molecules are unchanged, so a whole-fragment match ties assembly to TS.


In [ ]:
import itertools

XYZ = "structures/cpa.xyz"
ref = rx.read_xyz(XYZ, 0)
P = ref.GetConformer().GetPositions()
catalyst = "O=P1(O)Oc2c(-c3cccc4ccccc34)cc3ccccc3c2-c2c(c(-c3cccc4ccccc34)cc3ccccc23)O1"
substrate = "CC(=Cc1ccccc1)C(=O)C1=CCCCO1"
heavy_core, transfer_H = [23, 70, 74, 75], 35  # O(P), C, C, O(subs), and the proton in flight

# both fragments are unchanged here, so each matches the TS graph whole
cat_ref = ref.GetSubstructMatch(Chem.MolFromSmiles(catalyst))
sub_ref = ref.GetSubstructMatch(Chem.MolFromSmiles(substrate))
ncat = Chem.MolFromSmiles(catalyst).GetNumAtoms()
ts_to_asm = {t: f for f, t in enumerate(cat_ref)} | {t: ncat + f for f, t in enumerate(sub_ref)}

asm = Chem.AddHs(Chem.MolFromSmiles(f"{catalyst}.{substrate}"))
core = {ts_to_asm[t]: tuple(P[t]) for t in heavy_core}
o23 = ts_to_asm[23]
core[next(n.GetIdx() for n in asm.GetAtomWithIdx(o23).GetNeighbors() if n.GetAtomicNum() == 1)] = tuple(P[transfer_H])
# The proton is 1.367 A from the phosphate O in the TS, where this molecule's own graph allows at most 1.00.
# The graft holds it anyway: it is a Kabsch restore after the embed, so the bounds matrix never overrules the
# core. What the matrix contradiction costs is a strained SEED, which the log reports as a bound crossover.

ens = rx.embed(asm, fix=core, n=6, stereo="free").minimize()
drift = max(
    abs(np.linalg.norm(pos[ts_to_asm[i]] - pos[ts_to_asm[j]]) - np.linalg.norm(P[i] - P[j]))
    for c in ens.ids
    for pos in [ens.mol.GetConformer(c).GetPositions()]
    for i, j in itertools.combinations(heavy_core, 2)
)
print(f"{ens.n} conformers | reacting core held to {drift:.4f} A")

# The reacting REGION is wider than the grafted atoms: their neighbours inherit the TS's own distortion, and
# the reference geometry fails the same checks at the same atoms. Name it, or the gate flags real physics.
region = sorted(set(core) | {n.GetIdx() for a in core for n in asm.GetAtomWithIdx(a).GetNeighbors()})
clean = [c for c in ens.ids if rx.geom_check.check(ens.mol, conf_id=c, frozen=region).ok()]
print(f"gate: {len(clean)}/{ens.n} clean")

overlay(ens, ref, XYZ, [(ts_to_asm[t], t) for t in heavy_core])

## 2 · Mutate a fragment

A 70-atom frustrated Lewis pair TS: a hydride in flight between two borons. Once a fragment changes it no
longer matches the TS whole, so anchor SMARTS on the conserved motif locate the reacting atoms instead.
Three mutations: swap the substrate, swap the catalyst, swap both. The B-B and B-H distances hold in each.


In [ ]:
XYZ = "structures/FLP_TS.xyz"
ref = rx.read_xyz(XYZ, 0)
P = ref.GetConformer().GetPositions()

# anchors on the conserved motifs, so they survive mutating the periphery
BORANE_B = "[#5;$([#5]c);!$([#5]O)]"  # aryl-borane boron, not bonded to O
BORONATE_B = "[#5;$([#5]O)]"  # pinacol boronate boron
IPSO_C = "[#5;!$([#5]O)]c"  # borane boron plus its ipso carbon, a 4th atom to orient the graft
TS = {"bB": 11, "oB": 32, "hyd": 27, "ipso": 1}  # their positions in the TS

MUTATIONS = {
    "substrate (thienyl->phenyl)": ("Bc1ccccc1N1C(C)(C)CCCC1(C)C", "CC1(C)OB(c2ccccc2)OC1(C)C"),
    "catalyst  (TMP->piperidine)": ("Bc1ccccc1N1CCCCC1", "CC1(C)OB(c2cccs2)OC1(C)C"),
    "both": ("Bc1ccccc1N1CCCCC1", "CC1(C)OB(c2ccccc2)OC1(C)C"),
}
for label, (cat, sub) in MUTATIONS.items():
    asm = Chem.AddHs(Chem.MolFromSmiles(f"{cat}.{sub}"))
    (bB,) = rx.match(asm, BORANE_B)
    (oB,) = rx.match(asm, BORONATE_B)
    ipso = rx.match(asm, IPSO_C)[1]
    hyd = next(n.GetIdx() for n in asm.GetAtomWithIdx(bB).GetNeighbors() if n.GetAtomicNum() == 1)

    core = {a: tuple(P[TS[k]]) for k, a in (("bB", bB), ("oB", oB), ("hyd", hyd), ("ipso", ipso))}
    ens = rx.embed(asm, fix=core, n=6, stereo="free").minimize().prune()
    bb = np.mean([T.GetBondLength(ens.mol.GetConformer(c), bB, oB) for c in ens.ids])
    bh = np.mean([T.GetBondLength(ens.mol.GetConformer(c), bB, hyd) for c in ens.ids])
    print(f"  {label:28s} {ens.n} confs | B-B {bb:.3f} (DFT 2.164) | B-H {bh:.3f} (DFT 1.226)")

    overlay(ens, ref, XYZ, [(bB, 11), (oB, 32), (hyd, 27), (ipso, 1)])

## 3 · Handedness

The graft carries the exact 3D core, so handedness is explicit. `chirality='retain'` reproduces the DFT
hand; `chirality='both'` also embeds the mirror core so a real energy can choose.


In [ ]:
cat, sub = "Bc1ccccc1N1CCCCC1", "CC1(C)OB(c2ccccc2)OC1(C)C"  # the both-swapped assembly
asm = Chem.AddHs(Chem.MolFromSmiles(f"{cat}.{sub}"))
bB = asm.GetSubstructMatch(Chem.MolFromSmarts(BORANE_B))[0]
oB = asm.GetSubstructMatch(Chem.MolFromSmarts(BORONATE_B))[0]
ipso = asm.GetSubstructMatch(Chem.MolFromSmarts(IPSO_C))[1]
hyd = next(nb.GetIdx() for nb in asm.GetAtomWithIdx(bB).GetNeighbors() if nb.GetAtomicNum() == 1)
core = {bB: tuple(P[11]), oB: tuple(P[32]), hyd: tuple(P[27]), ipso: tuple(P[1])}


def hand(mol, cid):
    p = mol.GetConformer(cid).GetPositions()
    a, b, c, d = (p[i] for i in (ipso, oB, bB, hyd))
    return int(np.sign(np.dot(np.cross(b - a, c - a), d - a)))


keep = rx.embed(asm, fix=core, n=6, stereo="free")
# a coordinate graft carries handedness, so the mirrored core is the other diastereomer.
# a distance/angle spec cannot express this: it is reflection-invariant.
mirror = rx.embed(asm, fix={i: (x, y, -z) for i, (x, y, z) in core.items()}, n=6, stereo="free")
print(f"retain: {keep.n} confs, core hand {sorted({hand(keep.mol, c) for c in keep.ids})}")
print(
    f"both  : + {mirror.n} mirror confs, core hand {sorted({hand(mirror.mol, c) for c in mirror.ids})}  "
    f"(opposite => the other diastereomer)"
)

## 4 · A charged TS

An isothiourea-catalysed sulfa-Michael addition, 83 atoms: a cationic acyl isothiouronium and a deprotonated
thiourea nucleophile, net neutral. The acyl is conserved, so it is grafted whole and a fresh isothiourea
backbone hangs off it. The forming C-S bond holds at 2.266 Å across four backbones, charges preserved.


In [ ]:
ref = rx.read_xyz("structures/thia-ma.xyz", 0)
P = ref.GetConformer().GetPositions()
NUC = "O=[N+]([O-])c1ccc([N-]C(=S)Nc2ccc([N+](=O)[O-])cc2)cc1"  # deprotonated bis(p-nitrophenyl)thiourea (conserved)
ACYL_SM, THIO_SM = (
    "C(=O)C=CC(F)(F)F",
    "NC(=S)N",
)  # conserved acyl (holds the reacting beta-C) + thiourea core (holds S47)
acyl_ts = ref.GetSubstructMatch(Chem.MolFromSmarts(ACYL_SM))  # the TS atoms of each conserved motif (correct order)
thio_ts = ref.GetSubstructMatch(Chem.MolFromSmarts(THIO_SM))

ISOTHIOURONIUMS = {  # the acyl (with its reacting beta-C) is HELD; only the isothiourea backbone changes
    "HyperBTM (original)": "CC(C)[C@@H]1Cn2c([s+]c3ccccc32)N(C(=O)/C=C/C(F)(F)F)[C@@H]1c1ccccc1",
    "BTM (drop the i-Pr)": "C1Cn2c([s+]c3ccccc32)N(C(=O)/C=C/C(F)(F)F)[C@@H]1c1ccccc1",
    "p-tolyl variant": "CC(C)[C@@H]1Cn2c([s+]c3ccccc32)N(C(=O)/C=C/C(F)(F)F)[C@@H]1c1ccc(C)cc1",
    # a nitro-benzothiazolium isothiourea, acylated at its Lewis-base imine N (-> N+, the acyl isothiouronium):
    "nitro-fused (N-acylated)": "[O-][N+]([O-])=C1C=Cc2[s+]c3n(c2=C1)C1CC([N+](C(=O)/C=C/C(F)(F)F)=3)c2ccccc21",
}
for label, itu in ISOTHIOURONIUMS.items():
    asm = Chem.AddHs(Chem.MolFromSmiles(f"{itu}.{NUC}"))
    n0 = Chem.MolFromSmiles(itu).GetNumAtoms()
    acyl_a = asm.GetSubstructMatch(Chem.MolFromSmarts(ACYL_SM))  # same SMARTS -> atoms corresponding to acyl_ts
    thio_a = tuple(n0 + i for i in Chem.MolFromSmiles(NUC).GetSubstructMatch(Chem.MolFromSmarts(THIO_SM)))
    core = {a: tuple(P[t]) for a, t in zip(acyl_a, acyl_ts)}  # graft the whole conserved acyl to its TS geometry...
    core.update({a: tuple(P[t]) for a, t in zip(thio_a, thio_ts)})  # ...and the thiourea core to its TS geometry
    ens = (
        rx.embed(asm, fix=core, n=12, stereo="free").minimize().prune()
    )  # more seeds: the strained charged rings drop some in the gate
    beta, s47 = asm.GetSubstructMatch(Chem.MolFromSmarts("[CH1]=[CH1]C(F)(F)F"))[1], thio_a[thio_ts.index(47)]
    dcs = np.mean([T.GetBondLength(ens.mol.GetConformer(c), beta, s47) for c in ens.ids])
    print(
        f"  {label:22s}: {ens.n} confs | forming C(beta)-S held {dcs:.3f} A (DFT 2.266) | net charge {Chem.GetFormalCharge(asm)}"
    )
    low = ens.lowest(1)  # geometric align on the conserved reactive core (acyl + thiourea), then overlay
    amap = list(zip(acyl_a, acyl_ts)) + list(zip(thio_a, thio_ts))
    rdMolAlign.AlignMol(low.mol, ref, prbCid=low.ids[0], refCid=0, atomMap=amap)
    display(
        xyzrender.render(  # original TS (blue) vs the swapped isothiouronium (red), superposed on the core
            xyzrender.load("structures/thia-ma.xyz"),
            mol_color=ORIG,
            overlay=low.dump(tempfile.mktemp(suffix=".xyz")),
            overlay_color=NEW,
            auto_align=False,
            no_hy=True,
        )
    )

## 5 · Two things that need care

- locate reacting atoms by a conserved anchor, not the whole fragment: a mutated fragment, or one
  whose bonds change at the TS, will not match the TS whole
- a metal needs perceiving into a graph before its ligands can be swapped by SMILES. `rx.read_xyz`
  does that, and surrogating the metal separates each ligand into its own fragment


In [ ]:
# A metal complex needs perceiving into a graph before a ligand can be swapped by SMILES.
# rx.read_xyz does that (xyzgraph, the perceive extra); how far it gets depends on the complex.
for name in ("ru-co", "mn-h2"):
    m = rx.read_xyz(f"structures/{name}.xyz")
    smi = Chem.MolToSmiles(m)
    print(f"{name}: {m.GetNumAtoms()} atoms | SMILES round-trips = {Chem.MolFromSmiles(smi) is not None}")

# Surrogating the metal strips the M-L bonds, so each ligand falls out as its own fragment.
# That is the handle for a swap: keep the fragments you want, write a new SMILES for the one you do not.
from rxembed.metal_core import surrogate_all_metals

base, _metals = surrogate_all_metals(rx.read_xyz("structures/ru-co.xyz"))
for f in Chem.GetMolFrags(base):
    print(f"  {len(f):3d} atoms  {Chem.MolFragmentToSmiles(base, atomsToUse=list(f))[:56]}")

## 6 · Search a mutant and overlay it

The both-mutated molecule embedded, MC-searched, and overlaid on the original DFT TS. The core coincides;
the swapped periphery is what differs.


In [ ]:
ref = rx.read_xyz("structures/FLP_TS.xyz", 0)
P = ref.GetConformer().GetPositions()
asm = Chem.AddHs(Chem.MolFromSmiles("Bc1ccccc1N1CCCCC1.CC1(C)OB(c2ccccc2)OC1(C)C"))  # the both-mutant, fresh
bB = asm.GetSubstructMatch(Chem.MolFromSmarts(BORANE_B))[0]
oB = asm.GetSubstructMatch(Chem.MolFromSmarts(BORONATE_B))[0]
ipso = asm.GetSubstructMatch(Chem.MolFromSmarts(IPSO_C))[1]
hyd = next(nb.GetIdx() for nb in asm.GetAtomWithIdx(bB).GetNeighbors() if nb.GetAtomicNum() == 1)
core = {bB: tuple(P[11]), oB: tuple(P[32]), hyd: tuple(P[27]), ipso: tuple(P[1])}

searched = (
    rx.embed(asm, fix=core, n=8, stereo="free").mc(preset="ensemble").minimize().prune()
)  # 'ensemble' is the constrained-run preset
print(
    f"both-mutated TS: {searched.n} MC conformers | B(sub)-B(cat) held "
    f"{np.mean([T.GetBondLength(searched.mol.GetConformer(c), bB, oB) for c in searched.ids]):.3f} A"
)
low = searched.lowest(1)  # geometric align on the reactive core, then overlay
rdMolAlign.AlignMol(low.mol, ref, prbCid=low.ids[0], refCid=0, atomMap=[(bB, 11), (oB, 32), (hyd, 27), (ipso, 1)])
xyzrender.render(  # original TS (blue) + MC-searched mutant (red), superposed on the reacting core
    xyzrender.load("structures/FLP_TS.xyz"),
    mol_color=ORIG,
    overlay=low.dump(tempfile.mktemp(suffix=".xyz")),
    overlay_color=NEW,
    auto_align=False,
    no_hy=True,
)